# 04 Within-Domain Causal Discovery: Biomarkers

This notebook investigates the causal structure within the biomarker domain.

Objectives

- inspect biomarker dependency structure
- identify informative biomarkers
- formulate causal assumptions
- perform causal discovery
- compare data-driven and expert-driven graphs

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append("..")

from utils.feature_engineering import engineer_level6_features
from utils.domain_analysis import inspect_domain_dependencies

PROJECT_ROOT = Path("..")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
from utils.visualization import visualize_hierarchical_variable_dag
print("Libraries loaded.")

In [ ]:
DATASET_NAME = "attica_balanced_1to1_filtered.csv"

df_filtered = pd.read_csv(PROCESSED_DIR / DATASET_NAME)

print("=" * 80)
print("LOADED BALANCED ATTICA DATASET")
print("=" * 80)

print("Shape:", df_filtered.shape)
display(df_filtered.head())

In [ ]:
cfg_biomarkers = engineer_level6_features(df_filtered)

biomarker_features = cfg_biomarkers["features"]
biomarker_vars = list(biomarker_features.columns)

print("=" * 80)
print("BIOMARKER DOMAIN VARIABLES")
print("=" * 80)

print("Number of biomarker variables:", len(biomarker_vars))
print(biomarker_vars)

display(biomarker_features.head())

In [ ]:
biomarker_variable_table = pd.DataFrame({
    "Variable": biomarker_vars
})

biomarker_variable_table.to_csv(
    TABLES_DIR / "biomarker_domain_variables.csv",
    index=False
)

display(biomarker_variable_table)

In [ ]:
biomarker_inspection = inspect_domain_dependencies(
    df=biomarker_features,
    cols=biomarker_vars,
    domain_name="Biomarkers"
)

In [ ]:
selected_biomarkers = [

    # Inflammatory biomarkers
    "Il_6",
    "CRP",
    "wbc",
    "amiloide",
    "Fibrinogen",
    "TNF_a",
    "Hcy",

    # Lipid biomarkers
    "Cholesterol",
    "LDL",
    "HDL",
    "TGL",
    "ApoA1",
    "ApoB",
    "ox_LDL",

    # Renal biomarkers
    "creatinine",
    "creat_clearence",
    "ourea",
    "uric",
]

In [ ]:
df_biomarkers = biomarker_features[selected_biomarkers].copy()

print("=" * 80)
print("SELECTED ORIGINAL BIOMARKERS")
print("=" * 80)

print("Variables:", len(selected_biomarkers))
display(df_biomarkers.head())

In [ ]:
selected_inspection = inspect_domain_dependencies(
    df=df_biomarkers,
    cols=selected_biomarkers,
    domain_name="Original Biomarkers",
   
)

## Biomarker causal assumptions

The biomarker-level analysis is treated as an exploratory causal discovery task within one biomedical domain.

The purpose is not to claim definitive causal mechanisms from observational data, but to construct a biologically plausible candidate causal graph among measured biomarkers.

Main assumptions:

1. **Observational setting**  
   The ATTICA biomarker measurements are observational baseline variables.

2. **Causal Markov assumption**  
   Conditional independencies in the data are assumed to reflect separation relations in the underlying causal graph.

3. **Faithfulness assumption**  
   Observed independencies are assumed not to be caused by exact parameter cancellation.

4. **Acyclic baseline approximation**  
   The baseline biomarker graph is approximated as acyclic.

5. **Biological ordering**  
   Inflammatory cytokines, lipid transport markers, oxidative lipid markers, and renal-function markers are grouped into biological tiers.

6. **Possible hidden confounding**  
   Because unmeasured biological and lifestyle factors may affect biomarkers, FCI is used in addition to PC.

In [ ]:
# =============================================================================
# BIOMARKER BIOLOGICAL TIERS
# =============================================================================

biomarker_tiers = {
    # upstream inflammatory cytokines
    "Il_6": 1,
    "TNF_a": 1,

    # downstream inflammatory / acute-phase markers
    "CRP": 2,
    "Fibrinogen": 2,
    "wbc": 2,
    "amiloide": 2,

    # lipid transport markers
    "Cholesterol": 3,
    "LDL": 3,
    "HDL": 3,
    "TGL": 3,
    "ApoA1": 3,
    "ApoB": 3,

    # oxidative lipid marker
    "ox_LDL": 4,

    # renal / systemic markers
    "creatinine": 5,
    "creat_clearence": 5,
    "ourea": 5,
    "uric": 5,

    # metabolic vascular marker
    "Hcy": 5,
}

# Keep only variables available in selected dataset
biomarker_tiers = {
    k: v for k, v in biomarker_tiers.items()
    if k in df_biomarkers.columns
}

print("Biomarker tiers:")
for var, tier in sorted(biomarker_tiers.items(), key=lambda x: x[1]):
    print(f"Tier {tier}: {var}")

In [ ]:
# =============================================================================
# BIOMARKER BACKGROUND KNOWLEDGE
# =============================================================================

from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
from causallearn.graph.GraphNode import GraphNode

bio_names = list(df_biomarkers.columns)
bio_nodes = {name: GraphNode(name) for name in bio_names}

bk_bio = BackgroundKnowledge()

# Later biological tiers cannot point to earlier tiers
for source in bio_names:
    for target in bio_names:
        if source == target:
            continue

        source_tier = biomarker_tiers.get(source, 99)
        target_tier = biomarker_tiers.get(target, 99)

        if source_tier > target_tier:
            bk_bio.add_forbidden_by_node(
                bio_nodes[source],
                bio_nodes[target]
            )

print("✓ Biomarker background knowledge created")
print("Variables:", len(bio_names))

In [ ]:
# =============================================================================
# PREPARE BIOMARKER DATA FOR CAUSAL DISCOVERY
# =============================================================================

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

X_bio = df_biomarkers.copy()

imputer = SimpleImputer(strategy="median")
X_bio_imp = imputer.fit_transform(X_bio)

scaler = StandardScaler()
X_bio_scaled = scaler.fit_transform(X_bio_imp)

df_bio_model = pd.DataFrame(
    X_bio_scaled,
    columns=bio_names,
    index=df_biomarkers.index
)

bio_data = df_bio_model.values.astype(float)

print("Biomarker causal discovery input:", df_bio_model.shape)
print("Any NaNs:", np.isnan(bio_data).any())
display(df_bio_model.head())

In [ ]:
# =============================================================================
# PC ALGORITHM - BIOMARKER LEVEL
# =============================================================================

from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz
from utils.graph_utils import extract_directed_edges, print_edges

cg_pc_bio = pc(
    bio_data,
    alpha=0.05,
    indep_test=fisherz,
    stable=True,
    background_knowledge=bk_bio,
    node_names=bio_names,
    show_progress=True
)

pc_bio_edges = extract_directed_edges(
    cg_pc_bio.G,
    bio_names
)

print_edges(pc_bio_edges, "PC Biomarker Directed Edges")

In [ ]:
from utils.graph_utils import (
    check_cycles,
    transitive_reduce_edges,
    graph_statistics
)

G_pc_bio = check_cycles(
    pc_bio_edges,
    graph_name="PC Biomarker Graph",
    node_names=bio_names
)

display(graph_statistics(
    pc_bio_edges,
    graph_name="PC Biomarker Graph",
    node_names=bio_names
))

pc_bio_edges_reduced = transitive_reduce_edges(
    pc_bio_edges,
    graph_name="PC Biomarker Graph",
    node_names=bio_names
)

print_edges(pc_bio_edges_reduced, "PC Biomarker Transitive-Reduced Edges")

In [ ]:
# =============================================================================
# FCI ALGORITHM - BIOMARKER LEVEL
# =============================================================================

from causallearn.search.ConstraintBased.FCI import fci

cg_fci_bio = fci(
    bio_data,
    indep_test_method=fisherz,
    alpha=0.05,
    depth=-1,
    max_path_length=-1,
    background_knowledge=bk_bio,
    node_names=bio_names,
    show_progress=True
)

fci_graph_bio = cg_fci_bio[0] if isinstance(cg_fci_bio, tuple) else cg_fci_bio.G

fci_bio_edges = extract_directed_edges(
    fci_graph_bio,
    bio_names
)

print_edges(fci_bio_edges, "FCI Biomarker Directed Edges")

In [ ]:
G_fci_bio = check_cycles(
    fci_bio_edges,
    graph_name="FCI Biomarker Graph",
    node_names=bio_names
)

display(graph_statistics(
    fci_bio_edges,
    graph_name="FCI Biomarker Graph",
    node_names=bio_names
))

fci_bio_edges_reduced = transitive_reduce_edges(
    fci_bio_edges,
    graph_name="FCI Biomarker Graph",
    node_names=bio_names
)

print_edges(fci_bio_edges_reduced, "FCI Biomarker Transitive-Reduced Edges")

In [ ]:
GRAPHS_DIR = RESULTS_DIR / "graphs"
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

from utils.graph_utils import save_edges

save_edges(pc_bio_edges, GRAPHS_DIR / "pc_biomarker_edges.csv")
save_edges(pc_bio_edges_reduced, GRAPHS_DIR / "pc_biomarker_edges_reduced.csv")
save_edges(fci_bio_edges, GRAPHS_DIR / "fci_biomarker_edges.csv")
save_edges(fci_bio_edges_reduced, GRAPHS_DIR / "fci_biomarker_edges_reduced.csv")

print("Biomarker edge files saved.")

In [ ]:
summary_bio = pd.DataFrame({
    "Graph": [
        "PC biomarker",
        "PC biomarker reduced",
        "FCI biomarker",
        "FCI biomarker reduced",
    ],
    "Edges": [
        len(pc_bio_edges),
        len(pc_bio_edges_reduced),
        len(fci_bio_edges),
        len(fci_bio_edges_reduced),
    ]
})

display(summary_bio)

summary_bio.to_csv(
    TABLES_DIR / "biomarker_causal_discovery_summary.csv",
    index=False
)

In [ ]:
# ============================================================
# Biomarker hierarchy (based on biological knowledge)
# ============================================================

upstream_bio = [
    "Il_6",
    "TNF_a",
    "CRP",
    "Fibrinogen",
    "wbc",
    "amiloide",
]

middle_bio = [
    "Cholesterol",
    "LDL",
    "HDL",
    "TGL",
    "ApoB",
    "ApoA1",
    "ox_LDL",
]

downstream_bio = [
    "creatinine",
    "creat_clearence",
    "ourea",
    "uric",
    "Hcy",
]

In [ ]:
visualize_hierarchical_variable_dag(
    edges=pc_bio_edges_reduced,
    upstream_vars=upstream_bio,
    middle_vars=middle_bio,
    downstream_vars=downstream_bio,
    title="PC Algorithm - Biomarker-Level Causal Graph",
    filename=FIGURES_DIR / "pc_biomarker_dag_hierarchical.png",
)

In [ ]:
visualize_hierarchical_variable_dag(
    edges=fci_bio_edges_reduced,
    upstream_vars=upstream_bio,
    middle_vars=middle_bio,
    downstream_vars=downstream_bio,
    title="FCI Algorithm - Biomarker-Level Causal Graph",
    filename=FIGURES_DIR / "fci_biomarker_dag_hierarchical.png",
)

In [ ]:
pc_set = set(pc_bio_edges_reduced)
fci_set = set(fci_bio_edges_reduced)

shared_edges = pc_set.intersection(fci_set)
pc_only = pc_set.difference(fci_set)
fci_only = fci_set.difference(pc_set)

print("Shared edges:", len(shared_edges))
print("PC-only edges:", len(pc_only))
print("FCI-only edges:", len(fci_only))

edge_comparison = pd.DataFrame({
    "Category": ["Shared", "PC only", "FCI only"],
    "Count": [len(shared_edges), len(pc_only), len(fci_only)]
})

display(edge_comparison)

In [ ]:
# ============================================================
# LOAD DOMAIN-LEVEL DATASET
# ============================================================

from pathlib import Path
import pandas as pd

DOMAIN_DATASET_NAME = "attica_domain_scores.csv"

domain_df = pd.read_csv(
    PROCESSED_DIR / DOMAIN_DATASET_NAME
)

print("=" * 80)
print("DOMAIN-LEVEL ANALYTICAL DATASET")
print("=" * 80)

print("Dataset:", DOMAIN_DATASET_NAME)
print("Shape:", domain_df.shape)
print("\nColumns:")
print(domain_df.columns.tolist())

display(domain_df.head())

In [ ]:
# ============================================================
# DEFINE DOMAIN-LEVEL CAUSAL QUESTION
# ============================================================

TREATMENT = "Treatment"
OUTCOME = "Outcomes"

CONFOUNDERS = [
    "Demographics",
    "SES",
]

POTENTIAL_MEDIATORS = [
    "Obesity",
    "Metabolic",
    "Biomarkers",
    "Conditions",
]

analysis_columns = [
    TREATMENT,
    OUTCOME,
    *CONFOUNDERS,
    *POTENTIAL_MEDIATORS,
]

analysis_columns = list(dict.fromkeys(analysis_columns))

missing_columns = [
    column
    for column in analysis_columns
    if column not in domain_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required domain columns: {missing_columns}"
    )

df_causal = domain_df[analysis_columns].copy()

print("=" * 80)
print("DOMAIN-LEVEL CAUSAL QUESTION")
print("=" * 80)

print("Treatment:", TREATMENT)
print("Outcome:", OUTCOME)
print("Adjustment set:", CONFOUNDERS)
print("Potential mediators:", POTENTIAL_MEDIATORS)
print("Shape:", df_causal.shape)